# CeTTa from Python

Create an engine, load definitions and call them with Python values. Answers keep duplicate results and remain readable after the engine closes.


In [ ]:
from cetta import Engine, Symbol, CettaError
engine = Engine()


In [ ]:
engine.run("(= (double $x) (* $x 2))")
answers = engine.call("double", 21)
assert answers.to_python() == [42]
answers


A document can contain several evaluation directives. Each one has its own answer group.


In [ ]:
answers = engine.run("(= (coin) heads) (= (coin) heads) (= (coin) tails) !(coin) !(double 3)")
assert [len(group) for group in answers.groups] == [3, 1]
assert answers.to_python().count(Symbol("heads")) == 2
answers.groups


Load files directly. Relative imports use the source file’s directory; no process directory change is needed. Use `import_` for MeTTa’s `import!` operation.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as directory, Engine() as files:
    root = Path(directory)
    (root / "helper.metta").write_text("(= (triple $x) (* $x 3))", encoding="utf-8")
    source = root / "main.metta"
    source.write_text('!(import! &self "./helper.metta") !(triple 14)', encoding="utf-8")
    assert files.load(source)[-1].to_python() == 42
    files.import_(source)
    assert files.call("triple", 4).to_python() == [12]


Atoms are structured data. Reusing a variable preserves sharing; a new variable can have the same display name and a different identity.


In [ ]:
x = engine.variable("x")
y = engine.variable("x")
pair = engine.expression(Symbol("pair"), x, x)
assert pair.children[1] == pair.children[2]
assert x != y
pair


In [ ]:
unknown = engine.eval("(a-name-without-an-operation 42)")[0]
assert unknown.to_python() == (Symbol("a-name-without-an-operation"), 42)
unknown


Import nonstandard libraries explicitly. Python strings arrive as strings.


In [ ]:
engine.import_("(library str)")
assert engine.call("str:starts-with", "hello", "he")[0].as_bool()


In [ ]:
retained = engine.call("double", 5)[0]
engine.close()
assert retained.to_python() == 10
try:
    engine.eval("(+ 1 2)")
except CettaError as error:
    assert error.status == "engine-closed"
retained


PeTTa queries can stream results and their variable bindings. Close a stream when stopping early.

In [ ]:
with Engine() as streaming:
    streaming.run('(= (pick a) 11) (= (pick b) 22)')
    with streaming.query('(pick $x)') as query:
        rows = [(answer.to_python(), answer.bindings['x'].to_python()) for answer in query]
    assert rows == [(11, Symbol('a')), (22, Symbol('b'))]
rows

HE uses the same Python interface with its own language rules.


In [ ]:
with Engine(language="he") as he:
    he.run("(= (double $x) (* $x 2))")
    with he.query("(double (superpose (2 3)))") as answers:
        assert answers.to_python() == [4, 6]


Spaces hold facts. Queries return facts as data, together with scoped variable bindings. An explicit template computes a result from each match.

In [ ]:
with Engine() as knowledge:
    facts = knowledge.new_space()
    facts.add_atom(knowledge.parse("(temperature Prague 18)"))
    facts.add_atom(knowledge.parse("(temperature Brno 20)"))
    assert len(facts) == 2
    with facts.match("(temperature $city $degrees)") as query:
        rows = [(a.bindings["city"].to_python(), a.bindings["degrees"].to_python()) for a in query]
    assert rows == [(Symbol("Prague"), 18), (Symbol("Brno"), 20)]
    with facts.match("(temperature $city $degrees)", "(+ $degrees 1)") as query:
        assert query.to_python() == [19, 21]
rows


Host callbacks declare their arity and multiplicity. Returned code-shaped values stay data; generator answers can be streamed and stopped early.

In [ ]:
from cetta import Callback
with Engine() as callbacks:
    callbacks.register_callback("host:twice", Callback(lambda x: x * 2, 1))
    assert callbacks.call("host:twice", 21).to_python() == [42]
    callbacks.register_callback("host:range", Callback(lambda n: range(n), 1, generator=True))
    with callbacks.query("(host:range 4)") as query:
        assert query.batch(2).to_python() == [0, 1]
    assert callbacks.eval("(+ 1 2)").to_python() == [3]


NumPy inputs become CeTTa tensors. Mutable arrays are copied into stable values. Native outputs expose retained read-only views; request copy=True for a writable independent array.

In [ ]:
import numpy as np
with Engine() as numerical:
    source = np.arange(6, dtype=np.float64)
    tensor = numerical.tensor(source)
    view = tensor.to_numpy(copy=False)
    source[0] = 100
    assert view[0] == 0 and not view.flags.writeable
    numerical.import_("(library tensor)")
    assert numerical.call("tensor:sum", tensor).to_python() == [15.0]
    foreign = np.asarray([2.0, 3.0], dtype=(">" if np.little_endian else "<") + "f8")
    foreign_tensor = numerical.tensor(foreign)
    assert foreign_tensor.to_numpy().dtype == foreign.dtype
    assert numerical.call("tensor:sum", foreign_tensor).to_python() == [5.0]
assert view.tolist() == [0, 1, 2, 3, 4, 5]


Sparse tensors keep absent cells distinct from stored zeros. Their component arrays are retained read-only views. Choose a fill value explicitly when a dense array is wanted.

In [ ]:
with Engine() as sparse_engine:
    tensor = sparse_engine.tensor_from_coo((2, 3), [0, 4], [0.0, 7.0])
    sparse = tensor.to_sparse()
    assert sparse.nnz == 2 and sparse.coordinates.tolist() == [0, 4]
    assert sparse.to_numpy(fill=-1).tolist() == [[0, -1, -1], [-1, 7, -1]]
    assert not sparse.values.flags.writeable
    sparse_engine.import_("(library tensor)")
    compressed = sparse_engine.call("tensor:convert", tensor, Symbol("csc"))[0].to_sparse()
    assert compressed.layout == "csc"
    assert sparse_engine.tensor(compressed, copy=False).to_sparse().nnz == 2
assert sparse.values.tolist() == [0.0, 7.0]
sparse
